# 🛣️ PART 7: LSTM - The Memory Highway

Now we arrive at the solution that revolutionized sequence modeling: Long Short-Term Memory (LSTM).

## The Core Insight

> **LSTM adds a CELL STATE - a highway that runs through time, carrying information with minimal interference.**

## Two State Vectors

An RNN has one hidden state: $h_t$ (short-term memory)

An LSTM has TWO states:
- $h_t$: Hidden state (short-term memory/"working memory")
- $c_t$: Cell state (long-term memory/"filing cabinet")

## The Cell State: A Gradient Highway

The key equation that saves LSTMs:

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

Notice: $c_t$ is a **linear** combination of $c_{t-1}$ and new info! No non-linear activation between time steps!

During backpropagation:

$$\frac{\partial c_t}{\partial c_{t-1}} = f_t \quad \text{(a value between 0 and 1)}$$

This means gradients can flow through the cell state with only **multiplicative** decay, not the repeated application of tanh derivatives!

<div align="center">
  <img src="assets/lstm.png" width="400" alt="LSTM Architecture">
</div>

## The Three Gates

### 🧠 Mental Model: The Filing Cabinet

```
Imagine you're writing a book review:

CELL STATE (c_t) = The filing cabinet (long-term memory)
HIDDEN STATE (h_t) = Your desk (working memory)

Processing the sentence: "I absolutely loved this movie!"

Word 1: "I" 
    → File under "subject" (cell state updates minimally)
    → Desk shows current word is "I"

Word 2: "absolutely" 
    → Intensity modifier, update desk (hidden state)
    → But don't file permanently yet - still gathering info

Word 3: "loved" 
    → CRITICAL! File under "sentiment = positive" (cell state)
    → This goes in the cabinet permanently

Word 4: "this" 
    → Minor, just update desk

Word 5: "movie" 
    → Topic, file under "topic = movie" (cell state)

At the end, the filing cabinet (cell state) contains:
- sentiment: positive
- topic: movie
- intensity: high

While your desk (hidden state) contains the current context.
```

## The Three Gates Explained

**1. Forget Gate: What to Remove from Long-Term Memory**
$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$
- Takes previous hidden state and current input
- Outputs values between 0 and 1 for each dimension of cell state
- 1 = "completely keep this", 0 = "completely forget this"

**2. Input Gate: What to Add to Long-Term Memory**
$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$
$$\tilde{c}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)$$
- $i_t$: Decides which dimensions to update (0 or 1)
- $\tilde{c}_t$: Proposes new values to add (-1 to 1)

**3. Output Gate: What to Reveal in Working Memory**
$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$
$$h_t = o_t \odot \tanh(c_t)$$
- Decides what part of long-term memory to use for current output

## Building an LSTM Cell from Scratch

Let's implement this to truly understand each gate:

In [2]:
# ⚙️ Initial Setup - Run this first
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import Counter
import re
import warnings
import requests
from pathlib import Path
import time
warnings.filterwarnings('ignore')

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("✅ Imports loaded successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Create data directory
Path("../data/imdb").mkdir(parents=True, exist_ok=True)

✅ Imports loaded successfully!
PyTorch version: 2.10.0+cu128
GPU available: True
Using device: cuda


In [ ]:
class LSTMCell(nn.Module):

    """Single LSTM cell - built from scratch to understand each gate
    
    This implements the full LSTM equations:
    f_t = σ(W_f · [h_{t-1}, x_t] + b_f)
    i_t = σ(W_i · [h_{t-1}, x_t] + b_i)
    g_t = tanh(W_g · [h_{t-1}, x_t] + b_g)
    o_t = σ(W_o · [h_{t-1}, x_t] + b_o)
    c_t = f_t ⊙ c_{t-1} + i_t ⊙ g_t
    h_t = o_t ⊙ tanh(c_t)
    """

    def __init__(self, input_size, hidden_size): 
        super().__init() 

        self.input_size = input_size 
        self.hidden_size = hidden_size

        # we well have 4 gates : input , forget , cell , output
        self.W_ih = nn.Linear(input_size , 4*hidden_size)
        self.W_hh = nn.Linear(hidden_size , 4*hidden_size) 

    def forawrd(self, x , state):

        h_prev , c_prev = state 

        gates = self.W_ih(x) + self.W_hh(h_prev) 

        i,f,g,o = gates.chunck(4,dim=1) 

        i = torch.sigmoid(i)  # input gate : what to write
        f = torch.sigmoid(f) # forget gate : what to del 
        g = torch.tanh(g)   # cell gate : new candidate 
        o = torch.sigmoid(o) # output gate : what to output 

        c_next = o * torch.tanh(c_next) 

        h_next = o * torch.tanh(c_next) 

    def init(self , batch):

        h  = torch.zeros(batch , self.hidden_size)
        c = torch.zeros(batch , self.hidden_size) 
        return (h , c )



In [4]:
# Update this path to where your Harry Potter texts are stored
path_to_data = "/home/silva/SILVA.AI/Projects/SAIR/4_PyTorch/3_Sequence_and_NLP/harry_potter_txt"

# If the folder doesn't exist or is empty or has no .txt files, use sample
def _load_texts_from_folder(folder):
    if not os.path.isdir(folder):
        print(f"⚠️ Path not found or not a directory: {folder}")
        return None

    files = sorted([
        f for f in os.listdir(folder)
        if os.path.isfile(os.path.join(folder, f)) and f.lower().endswith(".txt")
    ])
    if not files:
        print(f"⚠️ No .txt files found in: {folder}")
        return None

    texts = []
    for fname in files:
        p = os.path.join(folder, fname)
        try:
            with open(p, "r", encoding="utf-8", errors="ignore") as f:
                raw = f.read()
        except Exception as e:
            print(f"⚠️ Failed to read {p}: {e}")
            continue

        # Remove lines that contain "Page" and normalize whitespace
        lines = [line for line in raw.splitlines() if "Page" not in line]
        cleaned = " ".join(" ".join(lines).split())
        if cleaned:
            texts.append(cleaned)

    return " ".join(texts) if texts else None

all_text = _load_texts_from_folder(path_to_data)
if all_text is None:
    print("⚠️ Harry Potter texts not usable. Using sample text for demonstration.")
    all_text = "Harry Potter was a wizard. He went to Hogwarts School of Witchcraft and Wizardry. " * 100
else:
    print(f"✅ Loaded text from: {path_to_data}")

print(f"Total characters: {len(all_text):,}")
print(f"First 200 chars:\n{all_text[:200]}")

⚠️ Path not found or not a directory: /home/silva/SILVA.AI/Projects/SAIR/4_PyTorch/3_Sequence_and_NLP/harry_potter_txt
⚠️ Harry Potter texts not usable. Using sample text for demonstration.
Total characters: 8,200
First 200 chars:
Harry Potter was a wizard. He went to Hogwarts School of Witchcraft and Wizardry. Harry Potter was a wizard. He went to Hogwarts School of Witchcraft and Wizardry. Harry Potter was a wizard. He went t


In [ ]:
uni_chars =  sorted(list(set(all_text)))
vocab_size = len(uni_chars)
char2idx = {c:i for i,c in enumerate(uni_chars)}
idx2char = {i:c for i,c in enumerate(uni_chars)}

test_char  = "H" 



H -> 2


In [14]:
class DataGenerator:
    """Samples random sequences from text for next-character prediction"""
    def __init__(self, text, seq_len=100):
        self.text = text
        self.seq_len = seq_len
        self.total_chars = len(text)

    def get_batch(self, batch_size):
        """Return a batch of (input, target) sequences"""
        inputs = []
        targets = []

        for _ in range(batch_size):
            # Pick random starting point
            start = np.random.randint(0, self.total_chars - self.seq_len - 1)
            end = start + self.seq_len
            
            # Input: sequence, Target: shifted by one
            input_seq = self.text[start:end]
            target_seq = self.text[start+1:end+1]
            
            # Convert to indices
            inputs.append([char2idx[c] for c in input_seq])
            targets.append([char2idx[c] for c in target_seq])

        return torch.tensor(inputs), torch.tensor(targets)

# Test it
data_gen = DataGenerator(all_text, seq_len=20)
x, y = data_gen.get_batch(2)
print(f"Input shape: {x.shape}")
print(f"Target shape: {y.shape}")

# Decode first sample
print(f"\nInput:  {''.join([idx2char[i.item()] for i in x[0]])}")
print(f"Target: {''.join([idx2char[i.item()] for i in y[0]])}")

Input shape: torch.Size([2, 20])
Target shape: torch.Size([2, 20])

Input:  of Witchcraft and Wi
Target: f Witchcraft and Wiz


In [ ]:
class LASTMGenerator(nn.Module):
    def __init__(self, vocab_size, emd_dim=128, hidden_size = 256,num_layers = 3 , dropout = 0.3):

        super().__init__()

        self.vocab_size = vocab_size
        self.emd_dim = emd_dim
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # 1- embed 
        self.embedding = nn.Embedding(vocab_size , emd_dim) 

        # 2- lstm layer 
        self.lstm = nn.LSTM(
            input_size=emd_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
 
         ) 
        
        # 3. output layer 
        self.fc = nn.Linear(hidden_size , vocab_size) 

    def forward(self, x, states = None):

        x = self.embedding(x) 

        if states is None: 
            h = torch.zeros(self.num_layers , x.size(0), self.hidden_size).to(x.device)

            c = torch.zeros(self.num_layers , x.size(0), self.hidden_size).to(x.device)

        output , (h_n , c_n) = self.lstm(x , states) 

        logits = self.fc(output) # batch_size , seq_len , vocab_size 

    
    def generate


        